# The emotion voice bank

Builds the reference clips the runtime clones from.

```
[emotion:amused] -> bank/amused.wav
[emotion:warm]   -> bank/warm.wav
[laugh]          -> bank/nv_laugh.wav
```

## Why it is built rather than prompted

Qwen3-TTS splits into three capabilities that do not combine the way you
would like:

* `1.7B-CustomVoice` accepts an `instruct` string - but only for nine preset
  timbres.
* `1.7B-VoiceDesign` accepts an `instruct` string and can invent a voice -
  but each generation re-derives it.
* `0.6B-Base` clones a voice from a reference clip - and the clone path
  never encodes an instruction at all.

So "a brand new voice" and "emotion control" cannot both live in one runtime
call. The instruction model is used here, once, and the result is baked into
the clips. In-context cloning preserves prosody better than x-vector-only
cloning, so the emotional delivery survives the clone.

## What comes out

| file | what |
|---|---|
| `voice_bank/<emotion>.wav` | one reference clip per delivery |
| `voice_bank/nv_<name>.wav` | the non-verbal clips, in the same voice |
| `voice_bank/bank.json` | the manifest: description, instructs, transcripts |

## Cost

One GPU, about ten minutes, and 4.5 GB of downloads. It is a build step: the
output is a few megabytes and ships with the OS.


In [ ]:
# --- dependencies -------------------------------------------------------------
# qwen-tts pins transformers 4.57.3, which is older than what mlx-lm wants.
# This notebook does not need mlx, so the pin is fine here.
%pip install -q --upgrade pip
%pip install -q "transformers==4.57.3" "huggingface_hub<1.0" qwen-tts soundfile numpy

import subprocess, sys
print(subprocess.run([sys.executable, "-c",
      "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__)"],
      capture_output=True, text=True).stdout)


In [ ]:
# --- configuration -------------------------------------------------------------
import os, sys, json, shutil, glob
from pathlib import Path

WORK = Path("/kaggle/working")
PIPE = WORK / "voicepipeline"
(PIPE / "tools").mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"

# The voice. The same description is used for every clip, so the timbre is
# identical across the bank; only the delivery directive changes.
VOICE_DESCRIPTION = (
    "A warm British male voice in his early thirties, low and unhurried, "
    "with a dry, self-aware sense of humour and a naturally gentle manner."
)

# The line every delivery clip speaks. Neutral enough to be delivered any
# way, and short enough to keep the clips small.
REFERENCE_LINE = (
    "Right, let's get this sorted. Tell me what you are seeing and we will "
    "work through it together."
)

OUT_DIR = WORK / "voice_bank"
OUT_DIR.mkdir(parents=True, exist_ok=True)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
# The pipeline is embedded verbatim, so it cannot drift from the repo.
# ---- tools/common.py -> tools/common.py -------------------------------------------------
from pathlib import Path as _P
_d = _P(WORK) / "voicepipeline" / "tools/common.py"
_d.parent.mkdir(parents=True, exist_ok=True)
_d.write_text(r'''
"""Shared constants, paths and helpers for the emotion-training toolchain.

Everything downstream (loader, annotator, leakage check, trainer, evaluator)
imports from here so the tag vocabulary and the canonical persona can never
drift between stages.
"""

from __future__ import annotations

import json
import os
import re
from pathlib import Path

# --------------------------------------------------------------------------
# Paths
# --------------------------------------------------------------------------

ROOT = Path(__file__).resolve().parent.parent

MODELS = ROOT / "models"
DATA = ROOT / "data"
ANNOTATED = DATA / "annotated"
BENCH = ROOT / "bench"

# Source artefacts that ship with the repo / LoRA zip.
LORA_DIR = MODELS / "llm" / "agents3_lora"
MERGED_DIR = MODELS / "llm" / "agents3_merged"
TEST_CONVERSATIONS = LORA_DIR / "test_conversations.json"

# Raw dataset sources.
DEEPSEEK_JSONL = ROOT / "deepseek_companion.jsonl"
HANDWRITTEN_SCRIPT = Path.home() / "Downloads" / "make_joyful_data.py"

# Canonical persona: extracted from the LoRA's own eval set so the system
# prompt is byte-identical to what the existing adapter was trained against.
PERSONA_PATH = DATA / "persona" / "system_prompt.txt"

# Derived model artefacts.
TEXT_BASE_DIR = MODELS / "llm" / "agents3_text_base"
MLX_BASE_DIR = MODELS / "mlx_base" / "Qwen3.5-0.8B-MLX-bf16"
MLX_ADAPTER_DIR = MODELS / "llm" / "agents3_mlx_adapter"
EMOTION_ADAPTER_DIR = MODELS / "llm" / "agents3_emotion_adapter"

# The teacher and the TTS voice bank.
TEACHER_DIR = MODELS / "teacher" / "Qwen3.5-9B-4bit"
VOICE_DIR = MODELS / "voice" / "bank"

# Teacher endpoint (mlx_lm server running the 9B).
TEACHER_BASE_URL = os.environ.get("TEACHER_BASE_URL", "http://127.0.0.1:8081/v1")
TEACHER_MODEL = os.environ.get("TEACHER_MODEL", "mlx-community/Qwen3.5-9B-4bit")

# --------------------------------------------------------------------------
# Tag vocabulary
# --------------------------------------------------------------------------

# Delivery tags. Turn-level: selects which reference clip the TTS clones from.
# Derived from the persona prompt's own rules:
#   "be supportive when the user is frustrated [warm],
#    focused during work [focused],
#    and playful when the stakes are low [amused]"
#   "never trivialize data loss, security, or serious problems [serious]"
DELIVERY_EMOTIONS = ("neutral", "amused", "warm", "focused", "serious")

# Non-verbal tags. Positional: splices a pre-rendered clip.
NONVERBALS = ("laugh", "chuckle", "sigh", "gasp", "hm", "erm")

# The emotion that must never carry a non-verbal. Data loss / security /
# real distress is a "no jokes" situation.
SERIOUS = "serious"

TAG_RE = re.compile(
    r"\[emotion:(?P<emotion>%s)\]" % "|".join(DELIVERY_EMOTIONS)
    + r"|\[(?P<nonverbal>%s)\]" % "|".join(NONVERBALS)
)

# Same, but anchored so we can tell "tag at start of turn" from "tag mid-text".
DELIVERY_TAG_RE = re.compile(r"\[emotion:(%s)\]" % "|".join(DELIVERY_EMOTIONS))

# Affect variants used to build the augmentation set (Mode B).
AFFECT_VARIANTS = ("frustrated", "delighted", "urgent", "confused", "neutral")

# Maximum words in a spoken reply (excluding tags).
MAX_SPOKEN_WORDS = 30

# Hard cap for the length gate; a few outliers are tolerated in the median.
MAX_SPOKEN_WORDS_P95 = 40


def strip_tags(text: str) -> str:
    """Remove every emotion/non-verbal tag, leaving spoken words only."""
    return TAG_RE.sub("", text).strip()


def tags_in(text: str) -> list[tuple[str, str]]:
    """Return every (kind, value) tag in order. kind is 'emotion' or 'nonverbal'."""
    out: list[tuple[str, str]] = []
    for m in TAG_RE.finditer(text):
        if m.group("emotion"):
            out.append(("emotion", m.group("emotion")))
        else:
            out.append(("nonverbal", m.group("nonverbal")))
    return out


def emotion_of(text: str) -> str | None:
    """Return the delivery emotion of a tagged reply, or None."""
    m = DELIVERY_TAG_RE.search(text)
    return m.group(1) if m else None


def nonverbals_of(text: str) -> list[str]:
    return [v for k, v in tags_in(text) if k == "nonverbal"]


def word_count(text: str) -> int:
    return len(strip_tags(text).split())


# --------------------------------------------------------------------------
# Persona
# --------------------------------------------------------------------------

_FALLBACK_PERSONA = (
    "You are a local-first desktop AI companion integrated into a custom Linux OS. "
    "Your personality is warm, loyal, intelligent, and quietly playful, with polished "
    "British-style wit and balanced dry humour."
)


def load_persona() -> str:
    """The canonical system prompt.

    Read from the LoRA's eval set (all 72 conversations share one prompt), which
    guarantees byte-identity with what the existing adapter was trained against.
    Falls back to a short literal if the eval set is unavailable.
    """
    if PERSONA_PATH.exists():
        return PERSONA_PATH.read_text(encoding="utf-8").strip()
    if TEST_CONVERSATIONS.exists():
        convs = json.loads(TEST_CONVERSATIONS.read_text(encoding="utf-8"))
        for conv in convs:
            for msg in conv:
                if msg["role"] == "system":
                    PERSONA_PATH.parent.mkdir(parents=True, exist_ok=True)
                    PERSONA_PATH.write_text(msg["content"].strip(), encoding="utf-8")
                    return msg["content"].strip()
    return _FALLBACK_PERSONA


# --------------------------------------------------------------------------
# Small utilities
# --------------------------------------------------------------------------


def write_jsonl(path: Path, records: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        for rec in records:
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> list[dict]:
    out = []
    with path.open(encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out


def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


# --------------------------------------------------------------------------
# Tokenisation
# --------------------------------------------------------------------------


def encode_ids(tokenizer, text: str) -> list[int]:
    """Token ids for a string, for either a HF tokenizer or an MLX wrapper.

    ``mlx_lm`` wraps the HF tokenizer in a ``TokenizerWrapper`` that is not
    itself callable, but keeps the original at ``_tokenizer``.
    """
    fn = tokenizer if callable(tokenizer) else getattr(tokenizer, "_tokenizer", None)
    if fn is None:
        raise TypeError(f"cannot encode with {type(tokenizer).__name__}")
    return fn(text, add_special_tokens=False)["input_ids"]


def build_tokenized_examples(
    records: list[dict], tokenizer, max_len: int = 1024
) -> list[dict]:
    """Tokenise chat records into (input_ids, labels) with the prompt masked.

    Only assistant tokens contribute to the loss. Training on the system prompt
    would spend capacity on tokens we never want the model to produce, and
    would teach it to *predict* tags rather than *generate* them.

    The prompt and the full text are tokenised separately, which can disagree
    at the boundary, so the longest common prefix is used to find where the
    assistant's turn actually starts.
    """
    out: list[dict] = []
    for rec in records:
        msgs = rec["messages"]
        prompt_text = tokenizer.apply_chat_template(
            msgs[:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
        full_text = prompt_text + msgs[-1]["content"] + "<|im_end|>\n"

        prompt_ids = encode_ids(tokenizer, prompt_text)
        full_ids = encode_ids(tokenizer, full_text)

        n = min(len(prompt_ids), len(full_ids))
        cut = 0
        while cut < n and prompt_ids[cut] == full_ids[cut]:
            cut += 1

        ids = full_ids[:max_len]
        labels = [-100] * cut + full_ids[cut:max_len]
        if all(l == -100 for l in labels):
            continue
        out.append({"input_ids": ids, "labels": labels})
    return out


def completion_mask(labels: list[list[int]]) -> list[list[int]]:
    """Turn -100 placeholders into a 0/1 mask for the loss."""
    return [[0 if v == -100 else 1 for v in row] for row in labels]
''', encoding="utf-8")
print("wrote", _d)

# ---- tools/design_voice.py -> tools/design_voice.py -------------------------------------------------
from pathlib import Path as _P
_d = _P(WORK) / "voicepipeline" / "tools/design_voice.py"
_d.parent.mkdir(parents=True, exist_ok=True)
_d.write_text(r'''
"""Design the voice, and bake the emotion into reference clips.

The problem this solves
-----------------------
Qwen3-TTS splits into three capabilities that do not combine the way you
would like:

  * ``1.7B-CustomVoice`` accepts an ``instruct`` string - but only for nine
    preset timbres.
  * ``1.7B-VoiceDesign`` accepts an ``instruct`` string and can invent a new
    voice - but each generation re-derives it.
  * ``0.6B-Base`` clones a voice from a reference clip - but the clone path
    never encodes an instruction at all, so the cloned voice has no emotion
    control whatsoever.

So "a brand new voice" and "emotion control" cannot both live in the same
runtime call. The way around it is to use the instruction model *at build
time*:

  1. ``1.7B-VoiceDesign`` renders the same voice description once per emotion,
     each with an emotion directive appended. Five clips, five deliveries, one
     timbre.
  2. The clips become the reference bank. At runtime ``0.6B-Base`` clones from
     whichever clip matches the turn's emotion.

ICL voice cloning preserves prosody better than x-vector-only cloning, so the
emotional delivery in the reference survives the clone. The result is real
emotional range on a CPU-capable model, with the cost paid once at build time
and about a megabyte of assets at runtime.

The consistency check
----------------------
A voice that drifts between the five clips is worse than no voice at all, so
the bank is verified with ECAPA speaker embeddings: the cosine similarity of
every clip against the mean must clear a threshold. Below it, the clips are
re-rendered.
"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from common import VOICE_DIR, write_json  # noqa: E402

# The five deliveries, matching the model's tag vocabulary exactly.
EMOTION_DIRECTIVES = {
    "neutral": "Plain and factual. No colour, no emphasis, just clear information.",
    "amused": "Dry, understated British humour. Warmly amused, a hint of a smile, never loud.",
    "warm": "Kind and encouraging, a little slower, with genuine warmth.",
    "focused": "Brisk and businesslike. No banter, no flourishes, getting to the point.",
    "serious": "Calm, level and grave. No humour whatsoever. Weight on every word.",
}

# The reference line each clip speaks. It should be neutral enough to be
# delivered any way, and short enough to keep the clips small.
REFERENCE_LINE = (
    "Right, let's get this sorted. Tell me what you are seeing and we will "
    "work through it together."
)

# Non-verbal clips are rendered as short lines that *contain* the sound, then
# used verbatim when spliced into a reply. They have to be in the same voice
# as the delivery clips, or the splice is audible.
NONVERBAL_LINES = {
    "laugh": "Ha! Ha! That is brilliant, genuinely.",
    "chuckle": "Heh. Nice one, that.",
    "sigh": "Ohhh. Right then. Deep breath.",
    "gasp": "Ah! Well, that is unexpected.",
    "hm": "Hmm. Right, let me think about that.",
    "erm": "Erm. Let me see.",
}

SPEAKER_HINT = (
    "A warm British male voice in his early thirties, low and unhurried, "
    "with a dry, self-aware sense of humour and a naturally gentle manner."
)


def build_instruct(description: str, emotion: str) -> str:
    """One natural-language instruction covering both timbre and delivery.

    The timbre half is byte-identical across every clip, which is what keeps
    the voice consistent. Only the delivery half changes.
    """
    return (
        f"{description.strip()} "
        f"Delivery: {EMOTION_DIRECTIVES[emotion].strip()}"
    )


def render_bank(
    model_path: str,
    out_dir: Path,
    description: str,
    device: str = "mps",
    dtype: str = "float32",
) -> dict:
    """Render one reference clip per emotion with Qwen3-TTS 1.7B VoiceDesign."""
    import numpy as np
    import soundfile as sf
    import torch
    from qwen_tts import Qwen3TTSModel

    out_dir.mkdir(parents=True, exist_ok=True)

    torch_dtype = {
        "float32": torch.float32,
        "float16": torch.float16,
        "bfloat16": torch.bfloat16,
    }[dtype]

    print(f"loading {model_path} on {device}")
    model = Qwen3TTSModel.from_pretrained(model_path, device_map=device, dtype=torch_dtype)

    clips = {}
    for emotion, directive in EMOTION_DIRECTIVES.items():
        instruct = build_instruct(description, emotion)
        print(f"  rendering {emotion}...")
        wavs, sr = model.generate_voice_design(
            text=REFERENCE_LINE,
            language="English",
            instruct=instruct,
        )
        audio = np.asarray(wavs[0], dtype="float32")
        path = out_dir / f"{emotion}.wav"
        sf.write(str(path), audio, sr)
        clips[emotion] = {
            "file": f"{emotion}.wav",
            "sample_rate": int(sr),
            "duration_s": round(len(audio) / sr, 3),
            "instruct": instruct,
            "ref_text": REFERENCE_LINE,
        }
        print(f"    {len(audio) / sr:.2f}s -> {path}")

    # Non-verbal clips, same voice, same description.
    nonverbals = {}
    neutral_instruct = build_instruct(description, "neutral")
    for name, line in NONVERBAL_LINES.items():
        print(f"  rendering {name}...")
        wavs, sr = model.generate_voice_design(
            text=line,
            language="English",
            instruct=neutral_instruct,
        )
        audio = np.asarray(wavs[0], dtype="float32")
        path = out_dir / f"nv_{name}.wav"
        sf.write(str(path), audio, sr)
        nonverbals[name] = {
            "file": f"nv_{name}.wav",
            "sample_rate": int(sr),
            "duration_s": round(len(audio) / sr, 3),
            "ref_text": line,
        }
        print(f"    {len(audio) / sr:.2f}s -> {path}")

    manifest = {
        "model": model_path,
        "description": description,
        "reference_line": REFERENCE_LINE,
        "clips": clips,
        "nonverbals": nonverbals,
    }
    write_json(out_dir / "bank.json", manifest)
    print(f"-> {out_dir / 'bank.json'}")
    return manifest


def check_consistency(bank_dir: Path, threshold: float = 0.80) -> dict:
    """ECAPA speaker similarity of every clip against the bank mean.

    A voice that shifts between clips is worse than no voice, so this is a
    hard gate on the bank.
    """
    import numpy as np
    import soundfile as sf

    try:
        import sherpa_onnx
    except ImportError:
        return {
            "checked": False,
            "reason": "sherpa-onnx not installed - pip install sherpa-onnx",
        }

    manifest = json.loads((bank_dir / "bank.json").read_text(encoding="utf-8"))
    clips = manifest["clips"]

    # ECAPA-TDNN speaker embedding extractor.
    model_dir = Path.home() / ".cache" / "sherpa-onnx" / "speaker-embedding-models"
    onnx = next(model_dir.glob("*.onnx"), None) if model_dir.exists() else None
    if onnx is None:
        return {
            "checked": False,
            "reason": (
                "no ECAPA model found; download with "
                "sherpa-onnx-download speaker-embedding-models"
            ),
        }

    extractor = sherpa_onnx.SpeakerEmbeddingExtractor(
        str(onnx), num_threads=2, debug=False
    )

    vectors = {}
    for emotion, info in clips.items():
        audio, sr = sf.read(str(bank_dir / info["file"]), dtype="float32")
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != 16000:
            import librosa

            audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
        vectors[emotion] = extractor.extract(audio, sr=16000)

    # The non-verbal clips must also be the same voice, or a splice is
    # audible.
    for name, info in manifest.get("nonverbals", {}).items():
        audio, sr = sf.read(str(bank_dir / info["file"]), dtype="float32")
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != 16000:
            import librosa

            audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
        vectors[f"nv_{name}"] = extractor.extract(audio, sr=16000)

    mean = np.mean(list(vectors.values()), axis=0)
    sims = {
        emotion: float(np.dot(v, mean) / (np.linalg.norm(v) * np.linalg.norm(mean)))
        for emotion, v in vectors.items()
    }
    passed = min(sims.values()) >= threshold
    return {
        "checked": True,
        "threshold": threshold,
        "similarity": {k: round(v, 4) for k, v in sims.items()},
        "min": round(min(sims.values()), 4),
        "passed": passed,
    }


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument(
        "--model",
        default="Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
        help="the instruction-capable model used at build time",
    )
    ap.add_argument("--out", type=Path, default=VOICE_DIR)
    ap.add_argument("--description", default=SPEAKER_HINT)
    ap.add_argument("--device", default="mps")
    ap.add_argument("--dtype", default="float32")
    ap.add_argument("--threshold", type=float, default=0.80)
    ap.add_argument("--check-only", action="store_true")
    args = ap.parse_args()

    if not args.check_only:
        render_bank(args.model, args.out, args.description, args.device, args.dtype)

    result = check_consistency(args.out, args.threshold)
    print(json.dumps(result, indent=2))
    if result.get("checked") and not result.get("passed"):
        print(
            f"\nFAIL: the least similar clip scores {result['min']:.3f}, below "
            f"{args.threshold}. The voice drifts between deliveries - re-render "
            f"with a tighter description."
        )
        return 1
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
''', encoding="utf-8")
print("wrote", _d)


In [ ]:
# --- render the bank -----------------------------------------------------------
sys.path.insert(0, str(PIPE / "tools"))
import design_voice

# The module's own defaults point at its own tree; override the two that
# matter for this run.
design_voice.VOICE_DIR = OUT_DIR
design_voice.REFERENCE_LINE = REFERENCE_LINE
design_voice.SPEAKER_HINT = VOICE_DESCRIPTION

import time
t0 = time.time()
manifest = design_voice.render_bank(
    MODEL_ID,
    OUT_DIR,
    VOICE_DESCRIPTION,
    device="cuda",
    dtype="bfloat16",
)
print(f"rendered in {(time.time() - t0) / 60:.1f} min")


In [ ]:
# --- the voice has to be the same voice in every clip ---------------------------
# A voice that drifts between deliveries is worse than no voice, so the ECAPA
# speaker-embedding cosine of each clip against the bank mean has to clear
# the threshold. Below it, the clips are re-rendered with a tighter
# description.
import design_voice
import json

result = design_voice.check_consistency(OUT_DIR, threshold=0.80)
print(json.dumps(result, indent=2))

if result.get("checked") and not result.get("passed"):
    print("FAIL: the voice drifts between deliveries - re-render with a tighter description")
else:
    print("voice is consistent across the bank")


In [ ]:
# --- save ----------------------------------------------------------------------
import shutil, json
from pathlib import Path

# Everything the runtime needs, in one folder.
BUNDLE = WORK / "voice_bank_bundle"
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
BUNDLE.mkdir(parents=True)
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file():
        shutil.copy2(p, BUNDLE / p.name)

manifest = json.loads((OUT_DIR / "bank.json").read_text())
print("bank contents:")
for name, info in manifest["clips"].items():
    print(f"  {name:10s} {info['duration_s']:.2f}s  {info['file']}")
for name, info in manifest.get("nonverbals", {}).items():
    print(f"  nv_{name:7s} {info['duration_s']:.2f}s  {info['file']}")

total = sum(p.stat().st_size for p in BUNDLE.iterdir())
print(f"\ntotal {total / 1e6:.2f} MB in {BUNDLE}")
print("Download this folder - it is the voice bank.")
